# AlphaHVAC — Real-Time Simulation Dashboard

This notebook **loads the trained model** (`alphaHVAC_trained.pth`) and runs a live  
Gradio simulation that replays the exact test evaluation with animated, step-by-step graphs.

**Requirements before running:**
- `alphaHVAC_trained.pth` in the same folder as this notebook
- `Test_Optimized.csv` in the same folder (generated during training)
- Install dependencies (Cell 1)

**What you get:**
- The same 4 plots as the original notebook (Energy, Damper, Actions, Temp Error)
- Animated simulation: builds plot step-by-step at 20–40 iterations/sec
- Live metric dashboard (energy saving %, comfort change, damper mean, action counts)
- Controls: Start / Pause / Reset / Speed slider
- Final evaluation grades matching the original results


## Cell 1 — Install Dependencies

In [ ]:

import subprocess, sys

packages = ["gradio", "torch", "numpy", "pandas", "matplotlib", "scikit-learn"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ All packages ready.")

✅ All packages ready.


## Cell 2 — Core Imports & Global Constants

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")          
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import copy
import threading
import time
import os
import gradio as gr


STATE_SIZE  = 15
ACTION_SIZE = 3


COMFORT_THRESHOLD = 0.3534
ENERGY_THRESHOLD  = 0.2486


MODEL_PATH    = "/Users/pawanpahune/AIFA FINAL SUBMISSION/AIFA-Course-Project-AlphaHVAC-/Final Model/alphaHVAC_trained.pth"
TEST_CSV_PATH = "/Users/pawanpahune/AIFA FINAL SUBMISSION/AIFA-Course-Project-AlphaHVAC-/Final Model/Test_Optimized.csv"

print("✅ Imports complete.")
print(f"   PyTorch  : {torch.__version__}")
print(f"   Gradio   : {gr.__version__}")
print(f"   NumPy    : {np.__version__}")
print(f"   Pandas   : {pd.__version__}")

✅ Imports complete.
   PyTorch  : 2.7.1
   Gradio   : 5.36.2
   NumPy    : 1.26.4
   Pandas   : 2.3.1


## Cell 3 — HVAC Environment (exact copy from training notebook)

In [ ]:
class HVACEnv:
    

    def __init__(self, data_path, damper_step=0.025, lam=0.85,
                 comfort_threshold=None, energy_threshold=None):
        self.df                = pd.read_csv(data_path).reset_index(drop=True)
        self.damper_step       = damper_step
        self.lam               = lam
        self.comfort_threshold = comfort_threshold if comfort_threshold else COMFORT_THRESHOLD
        self.energy_threshold  = energy_threshold  if energy_threshold  else ENERGY_THRESHOLD
        self.max_index         = len(self.df) - 1
        self.damper_idx        = self.df.columns.get_loc("damper_position")
        self.airflow_idx       = self.df.columns.get_loc("airflow_current")
        self.room_idx          = self.df.columns.get_loc("room_temp")
        self.setpoint_idx      = self.df.columns.get_loc("setpoint")
        self.signal_idx        = self.df.columns.get_loc("thermal_signal")
        self.prev_damper       = None
        self.reset()

    def reset(self, start_idx=None):
        if start_idx is not None:
            self.idx = int(start_idx)
        else:
            valid = np.where(self.df["damper_position"].values > 0)[0]
            self.idx = int(np.random.choice(valid))
        self.current_damper = float(self.df.iloc[self.idx]["damper_position"])
        self.prev_damper    = self.current_damper
        return self._get_state()

    def _get_state(self):
        return self.df.iloc[self.idx].values.astype(np.float32)

    def step(self, action):
        self.prev_damper = self.current_damper
        if action == 0:   self.current_damper -= self.damper_step
        elif action == 2: self.current_damper += self.damper_step
        self.current_damper = np.clip(self.current_damper, 0.0, 1.0)
        self.idx += 1
        done = self.idx >= self.max_index
        if done: self.idx = self.max_index
        row        = self.df.iloc[self.idx]
        base       = max(row["damper_position"], 1e-3)
        ratio      = np.clip(self.current_damper / base, 0.2, 2.0)
        airflow    = np.clip(row["airflow_current"] * ratio, 0.0, 1.0)
        energy     = airflow * row["thermal_signal"]
        temp_error = abs(row["room_temp"] - row["setpoint"])
        smooth     = 0.1 * abs(self.current_damper - self.prev_damper)
        reward     = -temp_error - self.lam * energy - smooth
        comfortable = temp_error < self.comfort_threshold
        wasteful    = energy > self.energy_threshold
        if   comfortable and wasteful     and action == 0: reward += 0.5
        elif comfortable and wasteful     and action == 2: reward -= 0.5
        elif (not comfortable)            and action == 2: reward += 0.3
        elif (not comfortable)            and action == 0: reward -= 0.3
        elif comfortable and not wasteful and action == 1: reward += 0.1
        next_state                  = row.values.astype(np.float32).copy()
        next_state[self.damper_idx] = self.current_damper
        return next_state, reward, done, {"energy": energy, "temp_error": temp_error}



if os.path.exists(TEST_CSV_PATH):
    _env = HVACEnv(TEST_CSV_PATH)
    print(f"✅ HVACEnv OK  |  {len(_env.df)} test rows  "
          f"|  comfort_threshold={_env.comfort_threshold:.4f}  "
          f"|  energy_threshold={_env.energy_threshold:.4f}")
else:
    print(f"⚠️  {TEST_CSV_PATH} not found — place it in the same folder as this notebook.")

✅ HVACEnv OK  |  1724 test rows  |  comfort_threshold=0.3534  |  energy_threshold=0.2486


## Cell 4 — Neural Network (exact copy from training notebook)

In [ ]:
class AlphaThermalNet(nn.Module):
   

    def __init__(self):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(STATE_SIZE, 128), nn.LayerNorm(128), nn.LeakyReLU(0.01), nn.Dropout(0.2),
            nn.Linear(128, 128),        nn.LayerNorm(128), nn.LeakyReLU(0.01), nn.Dropout(0.2),
            nn.Linear(128, 128),        nn.LayerNorm(128), nn.LeakyReLU(0.01), nn.Dropout(0.2),
        )
        self.policy_head = nn.Sequential(
            nn.Linear(128, 64), nn.LeakyReLU(0.01), nn.Linear(64, ACTION_SIZE)
        )
        self.value_head = nn.Sequential(
            nn.Linear(128, 64), nn.LeakyReLU(0.01), nn.Linear(64, 1)
        )

    def forward(self, x):
        h = self.shared(x)
        return F.softmax(self.policy_head(h), dim=-1), torch.tanh(self.value_head(h))


print("✅ AlphaThermalNet class defined.")

✅ AlphaThermalNet class defined.


## Cell 5 — MCTS (exact copy from training notebook)

In [5]:
class MCTSNode:
    def __init__(self, state, parent=None, prior=1.0):
        self.state    = state
        self.parent   = parent
        self.children = {}
        self.visits   = 0
        self.value    = 0.0
        self.prior    = prior


class MCTS:
    """Exact replica — simulations=100, c_puct=1.0 matches evaluation in training notebook."""

    def __init__(self, env, model, simulations=50, c_puct=1.5):
        self.env         = env
        self.model       = model
        self.simulations = simulations
        self.c_puct      = c_puct
        self.actions     = [0, 1, 2]

    def search(self, root_state, add_noise=True):
        root = MCTSNode(root_state)
        rp   = self._get_policy(root_state)
        if add_noise:
            noise = np.random.dirichlet([0.3] * 3)
            rp    = 0.75 * rp + 0.25 * noise
        er = copy.deepcopy(self.env)
        for a in self.actions:
            et = copy.deepcopy(er)
            ns, _, _, _ = et.step(a)
            root.children[a] = MCTSNode(ns, parent=root, prior=rp[a])
        for _ in range(self.simulations):
            ec   = copy.deepcopy(self.env)
            node = root
            while node.children:
                action, node = self._select(node)
                _, _, done, _ = ec.step(action)
                if done: break
            if not node.children:
                pr = self._get_policy(ec._get_state())
                for a in self.actions:
                    et = copy.deepcopy(ec)
                    ns, _, _, _ = et.step(a)
                    node.children[a] = MCTSNode(ns, parent=node, prior=pr[a])
            self._backprop(node, self._evaluate(ec))
        visits = np.array(
            [root.children[a].visits if a in root.children else 0 for a in self.actions],
            dtype=np.float32
        )
        return int(np.argmax(visits))

    def _select(self, node):
        best, ba, bc = -np.inf, None, None
        for a, c in node.children.items():
            q = c.value / (c.visits + 1e-6)
            u = self.c_puct * c.prior * np.sqrt(node.visits + 1) / (1 + c.visits)
            if q + u > best:
                best, ba, bc = q + u, a, c
        return ba, bc

    def _get_policy(self, state):
        self.model.eval()
        with torch.no_grad():
            p, _ = self.model(torch.tensor(state, dtype=torch.float32).unsqueeze(0))
        return p.squeeze(0).numpy()

    def _evaluate(self, ec):
        self.model.eval()
        with torch.no_grad():
            _, v = self.model(torch.tensor(ec._get_state(), dtype=torch.float32).unsqueeze(0))
        return v.item()

    def _backprop(self, node, value):
        while node:
            node.visits += 1
            node.value  += value
            node = node.parent


print("✅ MCTS class defined.")

✅ MCTS class defined.


## Cell 6 — Load Trained Model

In [6]:
def load_model(path=MODEL_PATH):
    """Load the saved model weights. Returns model in eval mode."""
    net = AlphaThermalNet()
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Model file '{path}' not found.\n"
            f"Place alphaHVAC_trained.pth in the same folder as this notebook."
        )
    net.load_state_dict(torch.load(path, map_location="cpu"))
    net.eval()
    return net


# Load now — used by all subsequent cells
try:
    TRAINED_MODEL = load_model()
    print(f"✅ Model loaded from  '{MODEL_PATH}'")
    # Quick forward-pass to confirm architecture matches
    with torch.no_grad():
        _p, _v = TRAINED_MODEL(torch.zeros(1, STATE_SIZE))
    print(f"   Policy output shape : {_p.shape}")
    print(f"   Value  output shape : {_v.shape}")
    print(f"   Sample policy probs : {_p.numpy().round(4)}")
except FileNotFoundError as e:
    print(f"❌ {e}")
    TRAINED_MODEL = None

✅ Model loaded from  '/Users/pawanpahune/AIFA FINAL SUBMISSION/AIFA-Course-Project-AlphaHVAC-/Final Model/alphaHVAC_trained.pth'
   Policy output shape : torch.Size([1, 3])
   Value  output shape : torch.Size([1, 1])
   Sample policy probs : [[0.2738 0.4505 0.2756]]


## Cell 7 — Pre-compute Full Test Run (Reproducible Results)

We run the entire test evaluation **once** upfront (same as the original notebook)  
and store all arrays. The Gradio simulation then just replays these stored arrays,  
giving you the **identical results** to the original training notebook.

In [7]:
def run_full_evaluation(model, test_csv=TEST_CSV_PATH, mcts_sims=100, c_puct=1.0):
    """
    Exact replica of the evaluation loop in training Cell 8.
    Returns all arrays needed for the 4 plots.
    """
    print("Running full test evaluation (this may take a few minutes with MCTS sims=100)...")
    test_env  = HVACEnv(test_csv)
    mcts_eval = MCTS(test_env, model, simulations=mcts_sims, c_puct=c_puct)
    state     = test_env.reset(start_idx=0)   # always start from idx=0
    model.eval()

    baseline_energy    = []
    model_energy       = []
    baseline_temp_err  = []
    model_temp_err     = []
    model_damper       = []
    actions_taken      = []

    total_steps = len(test_env.df) - 1
    for step_i in range(total_steps):
        row = test_env.df.iloc[test_env.idx]
        baseline_energy.append(row["airflow_current"] * row["thermal_signal"])
        baseline_temp_err.append(abs(row["room_temp"] - row["setpoint"]))

        action = mcts_eval.search(state, add_noise=False)  # greedy at eval
        actions_taken.append(action)

        state, _, done, info = test_env.step(action)
        model_energy.append(info["energy"])
        model_damper.append(test_env.current_damper)
        model_temp_err.append(abs(
            test_env.df.iloc[test_env.idx]["room_temp"] -
            test_env.df.iloc[test_env.idx]["setpoint"]
        ))

        # Progress log every 200 steps
        if (step_i + 1) % 200 == 0:
            print(f"  Step {step_i+1}/{total_steps}")

        if done:
            break

    # ── Compute summary statistics (exactly as in the original notebook) ──
    actions_arr  = np.array(actions_taken)
    n_dec        = int((actions_arr == 0).sum())
    n_hold       = int((actions_arr == 1).sum())
    n_inc        = int((actions_arr == 2).sum())
    total        = len(actions_arr)
    avg_base_e   = float(np.mean(baseline_energy))
    avg_model_e  = float(np.mean(model_energy))
    avg_base_t   = float(np.mean(baseline_temp_err))
    avg_model_t  = float(np.mean(model_temp_err))
    energy_saving = (1 - avg_model_e / avg_base_e) * 100
    temp_change   = ((avg_model_t - avg_base_t) / (avg_base_t + 1e-8)) * 100
    mean_damper   = float(np.mean(model_damper))
    frac_zero     = sum(1 for d in model_damper if d < 0.01) / len(model_damper)

    results = {
        "baseline_energy"   : baseline_energy,
        "model_energy"      : model_energy,
        "baseline_temp_err" : baseline_temp_err,
        "model_temp_err"    : model_temp_err,
        "model_damper"      : model_damper,
        "actions_taken"     : actions_taken,
        # Stats
        "n_dec"         : n_dec,
        "n_hold"        : n_hold,
        "n_inc"         : n_inc,
        "total"         : total,
        "avg_base_e"    : avg_base_e,
        "avg_model_e"   : avg_model_e,
        "avg_base_t"    : avg_base_t,
        "avg_model_t"   : avg_model_t,
        "energy_saving" : energy_saving,
        "temp_change"   : temp_change,
        "mean_damper"   : mean_damper,
        "frac_zero"     : frac_zero,
    }

    print(f"\n✅ Evaluation complete — {total} timesteps")
    print(f"   Energy saving  : {energy_saving:.1f}%")
    print(f"   Comfort change : {temp_change:+.1f}%")
    print(f"   Damper mean    : {mean_damper:.3f}")
    print(f"   Actions  DEC={n_dec} ({n_dec/total*100:.1f}%)  "
          f"HOLD={n_hold} ({n_hold/total*100:.1f}%)  "
          f"INC={n_inc} ({n_inc/total*100:.1f}%)")
    return results


# ── Run evaluation (or skip if model not loaded) ──────────────────────────
if TRAINED_MODEL is not None and os.path.exists(TEST_CSV_PATH):
    EVAL_RESULTS = run_full_evaluation(TRAINED_MODEL)
else:
    EVAL_RESULTS = None
    print("⚠️  Skipping evaluation — model or test CSV missing.")

Running full test evaluation (this may take a few minutes with MCTS sims=100)...
  Step 200/1723
  Step 400/1723
  Step 600/1723
  Step 800/1723
  Step 1000/1723
  Step 1200/1723
  Step 1400/1723
  Step 1600/1723

✅ Evaluation complete — 1723 timesteps
   Energy saving  : 62.2%
   Comfort change : +0.0%
   Damper mean    : 0.793
   Actions  DEC=241 (14.0%)  HOLD=674 (39.1%)  INC=808 (46.9%)


## Cell 8 — Plot Builder (The 4-Panel Chart)

In [ ]:
# Color map for actions
ACTION_CMAP = {0: "#e74c3c", 1: "#f39c12", 2: "#27ae60"}


def build_plot(results: dict, step: int) -> plt.Figure:
    """
    Build the 4-panel figure showing data up to `step` (exclusive).
    step=None means show all data (final static view).
    """
    if step is None:
        step = results["total"]
    step = max(1, min(step, results["total"]))

    # Slice data up to current simulation step
    base_e   = results["baseline_energy"][:step]
    mod_e    = results["model_energy"][:step]
    base_t   = results["baseline_temp_err"][:step]
    mod_t    = results["model_temp_err"][:step]
    damper   = results["model_damper"][:step]
    actions  = results["actions_taken"][:step]
    steps_x  = range(step)

    # Live stats
    avg_be   = float(np.mean(base_e))  if base_e  else 0.0
    avg_me   = float(np.mean(mod_e))   if mod_e   else 0.0
    avg_bt   = float(np.mean(base_t))  if base_t  else 0.0
    avg_mt   = float(np.mean(mod_t))   if mod_t   else 0.0
    e_saving = (1 - avg_me / avg_be) * 100 if avg_be > 0 else 0.0
    t_change = ((avg_mt - avg_bt) / (avg_bt + 1e-8)) * 100 if avg_bt > 0 else 0.0
    mean_d   = float(np.mean(damper))  if damper  else 0.0
    frac_z   = (sum(1 for d in damper if d < 0.01) / len(damper)) * 100 if damper else 0.0

    n_dec  = actions.count(0)
    n_hold = actions.count(1)
    n_inc  = actions.count(2)
    tot    = len(actions) if actions else 1

    
    fig, axes = plt.subplots(4, 1, figsize=(14, 16))
    fig.suptitle(
        f"AlphaHVAC — Simulation  [{step}/{results['total']} steps]",
        fontsize=14, fontweight="bold"
    )

    # ── Panel 1: Energy
    ax = axes[0]
    ax.plot(base_e, label="Baseline",     color="steelblue",  lw=1.0)
    ax.plot(mod_e,  label="AlphaThermal", color="darkorange", lw=1.0)
    if len(steps_x) > 1:
        ax.fill_between(
            list(steps_x), base_e, mod_e,
            where=[b > m for b, m in zip(base_e, mod_e)],
            alpha=0.15, color="green", label="Energy saved"
        )
    ax.set_title(
        f"Energy — Saving: {e_saving:.1f}%  "
        f"(Base: {avg_be:.4f}  Model: {avg_me:.4f})",
        fontsize=11
    )
    ax.set_ylabel("Energy proxy", fontsize=10)
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(alpha=0.3)
    ax.set_xlim(0, results["total"])

    # ── Panel 2: Damper position
    ax = axes[1]
    ax.plot(damper, color="#8e44ad", lw=0.8, label="Damper position")
    ax.axhline(0, color="red",  linestyle="--", lw=1.0, label="Damper=0 (fully closed)")
    ax.axhline(1, color="blue", linestyle="--", lw=1.0, label="Damper=1 (fully open)")
    ax.set_title(
        f"Damper — mean={mean_d:.3f}  zero%: {frac_z:.1f}%",
        fontsize=11
    )
    ax.set_ylabel("Damper position (0–1)", fontsize=10)
    ax.set_ylim(-0.1, 1.1)
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(alpha=0.3)
    ax.set_xlim(0, results["total"])

    # ── Panel 3: Actions scatter
    ax = axes[2]
    if actions:
        ax.scatter(
            list(steps_x), actions,
            c=[ACTION_CMAP[a] for a in actions],
            s=3, alpha=0.7
        )
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(["0: DECREASE", "1: HOLD", "2: INCREASE"], fontsize=9)
    ax.set_title(
        f"Actions — DEC: {n_dec} ({n_dec/tot*100:.1f}%)   "
        f"HOLD: {n_hold} ({n_hold/tot*100:.1f}%)   "
        f"INC: {n_inc} ({n_inc/tot*100:.1f}%)",
        fontsize=11
    )
    patches = [
        mpatches.Patch(color=ACTION_CMAP[i], label=lbl)
        for i, lbl in [(0, "DEC"), (1, "HOLD"), (2, "INC")]
    ]
    ax.legend(handles=patches, fontsize=9, loc="upper right")
    ax.grid(alpha=0.3)
    ax.set_xlim(0, results["total"])

    # ── Panel 4: Temperature error
    ax = axes[3]
    ax.plot(base_t, label="Baseline",     color="steelblue",  lw=1.0)
    ax.plot(mod_t,  label="AlphaThermal", color="darkorange", lw=1.0)
    ax.set_title(
        f"Temp Error — Change: {t_change:+.1f}%  "
        f"(Base: {avg_bt:.4f}  Model: {avg_mt:.4f})",
        fontsize=11
    )
    ax.set_ylabel("|room_temp − setpoint|", fontsize=10)
    ax.set_xlabel("Timestep", fontsize=10)
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(alpha=0.3)
    ax.set_xlim(0, results["total"])

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


# ── Quick static test
if EVAL_RESULTS is not None:
    fig_test = build_plot(EVAL_RESULTS, step=200)
    plt.savefig("test_preview.png", dpi=80, bbox_inches="tight")
    plt.close(fig_test)
    print("✅ Plot builder OK — preview saved to test_preview.png")
else:
    print("⚠️  Skipping plot test — no evaluation results.")

✅ Plot builder OK — preview saved to test_preview.png


## Cell 9 — Simulation State Manager

In [ ]:
class SimState:
    """
    Thread-safe simulation state.
    Controls the background thread that advances the step counter.
    """

    def __init__(self, results: dict):
        self.results   = results
        self.total     = results["total"]
        self._step     = 1
        self._running  = False
        self._speed    = 20       
        self._lock     = threading.Lock()
        self._thread   = None

    # ── Thread-safe getters / setters 
    @property
    def step(self):
        with self._lock:
            return self._step

    @step.setter
    def step(self, v):
        with self._lock:
            self._step = int(np.clip(v, 1, self.total))

    @property
    def speed(self):
        with self._lock:
            return self._speed

    @speed.setter
    def speed(self, v):
        with self._lock:
            self._speed = max(1, int(v))

    @property
    def running(self):
        with self._lock:
            return self._running

    # ── Control methods
    def start(self):
        with self._lock:
            if self._running:
                return
            if self._step >= self.total:
                self._step = 1       
            self._running = True
        self._thread = threading.Thread(target=self._run_loop, daemon=True)
        self._thread.start()

    def pause(self):
        with self._lock:
            self._running = False

    def reset(self):
        with self._lock:
            self._running = False
            self._step    = 1

    def _run_loop(self):
        """Background thread: advance step counter at desired fps."""
        while True:
            with self._lock:
                if not self._running:
                    break
                self._step = min(self._step + self._speed, self.total)
                if self._step >= self.total:
                    self._running = False
                    break
            time.sleep(1.0)      



SIM = SimState(EVAL_RESULTS) if EVAL_RESULTS is not None else None
print("✅ SimState ready." if SIM else "⚠️  SimState skipped — no results.")

✅ SimState ready.


## Cell 10 — Evaluation Grading Function

In [10]:
def compute_grades(results: dict) -> str:
    """Exact grading logic from Cell 10 of the training notebook."""
    energy_saving = results["energy_saving"]
    temp_change   = results["temp_change"]
    mean_damp     = results["mean_damper"]
    frac_zero     = results["frac_zero"]
    n_dec, n_hold, n_inc, total = (
        results["n_dec"], results["n_hold"], results["n_inc"], results["total"]
    )

    # A: Energy
    grade_e = "GOOD" if energy_saving >= 20 else ("WEAK" if energy_saving >= 5 else "FAIL")

    # B: Comfort
    grade_t = "GOOD" if temp_change <= 5 else ("ACCEPTABLE" if temp_change <= 20 else "POOR")

    # C: Damper
    if frac_zero > 0.4:
        grade_d = "FAIL — damper stuck at zero"
    elif mean_damp > 0.85:
        grade_d = "FAIL — damper stuck at max"
    elif 0.2 <= mean_damp <= 0.75:
        grade_d = "GOOD — damper moves in sensible range"
    else:
        grade_d = "ACCEPTABLE"

    # D: Policy bias
    near_uniform   = (abs(n_dec/total - 1/3) < 0.05 and abs(n_inc/total - 1/3) < 0.05)
    max_action_pct = max(n_dec, n_hold, n_inc) / total * 100
    if near_uniform:
        grade_c = "FAIL — near-uniform (random policy)"
    elif max_action_pct > 75:
        dominant = ["DECREASE", "HOLD", "INCREASE"][int(np.argmax([n_dec, n_hold, n_inc]))]
        grade_c  = f"WEAK — {dominant} dominates ({max_action_pct:.0f}%)"
    else:
        grade_c = "GOOD — no single action dominates"

    # Overall verdict
    grades  = [grade_e, grade_t, grade_d, grade_c]
    n_good  = sum(g.startswith("GOOD") for g in grades)
    n_fail  = sum(g.startswith("FAIL") for g in grades)
    if n_fail == 0 and n_good >= 3:
        verdict = "✅ MODEL IS GENUINELY GOOD"
    elif n_fail == 0 and n_good >= 2:
        verdict = "⚠️  MODEL IS PARTIALLY WORKING"
    else:
        verdict = "❌ MODEL HAS ISSUES — see grades"

    lines = [
        "═" * 52,
        "  AlphaHVAC EVALUATION REPORT",
        "═" * 52,
        f"[A] ENERGY SAVING : {energy_saving:.1f}%  →  {grade_e}",
        f"    Base: {results['avg_base_e']:.4f}   Model: {results['avg_model_e']:.4f}",
        "",
        f"[B] COMFORT       : {temp_change:+.1f}%  →  {grade_t}",
        f"    Base: {results['avg_base_t']:.4f}   Model: {results['avg_model_t']:.4f}",
        "",
        f"[C] DAMPER        : mean={mean_damp:.3f}  zero%={frac_zero*100:.1f}%",
        f"    {grade_d}",
        "",
        f"[D] POLICY        : DEC={n_dec/total*100:.1f}%  HOLD={n_hold/total*100:.1f}%  INC={n_inc/total*100:.1f}%",
        f"    {grade_c}",
        "",
        "═" * 52,
        f"  {verdict}",
        "═" * 52,
    ]
    return "\n".join(lines)


if EVAL_RESULTS is not None:
    print(compute_grades(EVAL_RESULTS))
else:
    print("⚠️  No results to grade.")

════════════════════════════════════════════════════
  AlphaHVAC EVALUATION REPORT
════════════════════════════════════════════════════
[A] ENERGY SAVING : 62.2%  →  GOOD
    Base: 0.5344   Model: 0.2021

[B] COMFORT       : +0.0%  →  GOOD
    Base: 0.3801   Model: 0.3802

[C] DAMPER        : mean=0.793  zero%=0.9%
    ACCEPTABLE

[D] POLICY        : DEC=14.0%  HOLD=39.1%  INC=46.9%
    GOOD — no single action dominates

════════════════════════════════════════════════════
  ✅ MODEL IS GENUINELY GOOD
════════════════════════════════════════════════════


## Cell 11 — Gradio Dashboard

Run this cell to launch the interactive simulation dashboard.

In [ ]:
def launch_dashboard():
    if EVAL_RESULTS is None or SIM is None:
        print("❌ Cannot launch — evaluation results are missing.")
        print("   Make sure alphaHVAC_trained.pth and Test_Optimized.csv exist.")
        return

   
    def get_current_plot():
        """Returns the matplotlib figure for the current SIM step."""
        fig = build_plot(EVAL_RESULTS, step=SIM.step)
        return fig

    def get_metrics():
        """Returns a Markdown string of live metrics."""
        step = SIM.step
        r    = EVAL_RESULTS

        base_e_slice = r["baseline_energy"][:step]
        mod_e_slice  = r["model_energy"][:step]
        base_t_slice = r["baseline_temp_err"][:step]
        mod_t_slice  = r["model_temp_err"][:step]
        damper_slice = r["model_damper"][:step]
        act_slice    = r["actions_taken"][:step]

        avg_be  = np.mean(base_e_slice)  if base_e_slice  else 0.0
        avg_me  = np.mean(mod_e_slice)   if mod_e_slice   else 0.0
        avg_bt  = np.mean(base_t_slice)  if base_t_slice  else 0.0
        avg_mt  = np.mean(mod_t_slice)   if mod_t_slice   else 0.0
        esave   = (1 - avg_me / avg_be) * 100 if avg_be > 0 else 0.0
        tchange = ((avg_mt - avg_bt) / (avg_bt + 1e-8)) * 100
        md      = np.mean(damper_slice)  if damper_slice  else 0.0
        fz      = sum(1 for d in damper_slice if d < 0.01) / len(damper_slice) * 100 if damper_slice else 0.0
        tot     = len(act_slice) if act_slice else 1
        nd      = act_slice.count(0)
        nh      = act_slice.count(1)
        ni      = act_slice.count(2)
        status  = "▶ Running" if SIM.running else ("⏸ Paused" if step < r["total"] else "✅ Complete")

        return (
            f"### Live Metrics  —  Step **{step}** / {r['total']}  {status}\n"
            f"| Metric | Baseline | Model | Δ |\n"
            f"|--------|----------|-------|---|\n"
            f"| Energy | {avg_be:.4f} | {avg_me:.4f} | **{esave:.1f}% saved** |\n"
            f"| Temp error | {avg_bt:.4f} | {avg_mt:.4f} | {tchange:+.1f}% |\n"
            f"| Damper mean | — | {md:.3f} | zero: {fz:.1f}% |\n"
            f"| Actions | — | DEC {nd/tot*100:.0f}% / HOLD {nh/tot*100:.0f}% / INC {ni/tot*100:.0f}% | — |\n"
        )

    def btn_start(speed_val):
        SIM.speed = int(speed_val)
        SIM.start()
        time.sleep(0.05)              
        return get_current_plot(), get_metrics()

    def btn_pause():
        SIM.pause()
        return get_current_plot(), get_metrics()

    def btn_reset():
        SIM.reset()
        return get_current_plot(), get_metrics()

    def btn_refresh():
        """Manual refresh — used by the polling timer."""
        return get_current_plot(), get_metrics()

    def btn_final():
        """Jump to the final full result."""
        SIM.pause()
        SIM.step = EVAL_RESULTS["total"]
        return get_current_plot(), get_metrics(), compute_grades(EVAL_RESULTS)

    def btn_jump(step_val):
        """Jump to a specific step."""
        SIM.pause()
        SIM.step = int(step_val)
        return get_current_plot(), get_metrics()

    
    with gr.Blocks(
        title="AlphaHVAC Simulation",
        theme=gr.themes.Base(
            primary_hue="violet",
            secondary_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "sans-serif"]
        ),
        css="""
            .panel-box { border: 1px solid #e2e8f0; border-radius: 10px;
                         padding: 14px; background: #fafafa; }
            .btn-row   { display: flex; gap: 8px; flex-wrap: wrap; }
            h1         { text-align: center; font-size: 1.6rem; margin-bottom: 4px; }
            .subtitle  { text-align: center; color: #64748b; margin-bottom: 16px; }
        """
    ) as demo:

        gr.HTML(
            "<h1>🌡️ AlphaHVAC — Real-Time Simulation Dashboard</h1>"
            "<p class='subtitle'>AlphaZero-style HVAC damper control · "
            f"Test set: <b>{EVAL_RESULTS['total']}</b> timesteps · "
            f"Energy saved: <b>{EVAL_RESULTS['energy_saving']:.1f}%</b></p>"
        )


        with gr.Row():
            with gr.Column(scale=2, elem_classes=["panel-box"]):
                gr.Markdown("### 🎮 Simulation Controls")

                speed_slider = gr.Slider(
                    minimum=1, maximum=100, value=20, step=1,
                    label="Speed (steps rendered per second tick)",
                    info="20 = smooth, 40 = fast, 100 = near-instant"
                )
                jump_slider = gr.Slider(
                    minimum=1, maximum=EVAL_RESULTS["total"],
                    value=1, step=1,
                    label="Jump to step"
                )

                with gr.Row(elem_classes=["btn-row"]):
                    btn_start_btn  = gr.Button("▶ Start",    variant="primary")
                    btn_pause_btn  = gr.Button("⏸ Pause",    variant="secondary")
                    btn_reset_btn  = gr.Button("⏮ Reset",    variant="secondary")
                    btn_final_btn  = gr.Button("⏭ Full Run", variant="secondary")
                    btn_jump_btn   = gr.Button("⏩ Jump",     variant="secondary")
                    btn_refresh_btn= gr.Button("🔄 Refresh",  variant="secondary")

            with gr.Column(scale=3, elem_classes=["panel-box"]):
                metrics_md = gr.Markdown(
                    value=get_metrics(),
                    label="Live Metrics"
                )

       
        with gr.Row():
            chart_plot = gr.Plot(
                value=get_current_plot(),
                label="AlphaHVAC — 4-Panel Performance Chart"
            )

    
        with gr.Row():
            eval_report = gr.Textbox(
                value="",
                label="Evaluation Report (click 'Full Run' to generate)",
                lines=18,
                max_lines=20,
                interactive=False,
                show_copy_button=True
            )

        with gr.Accordion("ℹ️ About this simulation", open=False):
            gr.Markdown("""
            **How it works:**
            - The model (`alphaHVAC_trained.pth`) was trained using AlphaZero-style self-play
              on HVAC sensor data from building B90, room 102.
            - This dashboard **replays** the pre-computed test evaluation — no retraining.
            - The 4 charts show: Energy consumption, Damper position, Action distribution,
              and Temperature error — all vs. the baseline (original controller).

            **Chart colors:**
            - 🔵 Blue = baseline (original controller)
            - 🟠 Orange = AlphaHVAC model
            - 🟢 Green shading = energy saved
            - 🔴 Red dots = DECREASE action | 🟡 Yellow = HOLD | 🟢 Green = INCREASE

            **Key results (full test set):**
            - Energy saving: **{:.1f}%**
            - Comfort change: **{:+.1f}%** (near zero = maintained)
            - Damper mean: **{:.3f}** (dynamic, not stuck)
            """.format(
                EVAL_RESULTS["energy_saving"],
                EVAL_RESULTS["temp_change"],
                EVAL_RESULTS["mean_damper"]
            ))

        
        timer = gr.Timer(value=1.0)   

        btn_start_btn.click(
            fn=btn_start,
            inputs=[speed_slider],
            outputs=[chart_plot, metrics_md]
        )
        btn_pause_btn.click(
            fn=btn_pause,
            outputs=[chart_plot, metrics_md]
        )
        btn_reset_btn.click(
            fn=btn_reset,
            outputs=[chart_plot, metrics_md]
        )
        btn_final_btn.click(
            fn=btn_final,
            outputs=[chart_plot, metrics_md, eval_report]
        )
        btn_jump_btn.click(
            fn=btn_jump,
            inputs=[jump_slider],
            outputs=[chart_plot, metrics_md]
        )
        btn_refresh_btn.click(
            fn=btn_refresh,
            outputs=[chart_plot, metrics_md]
        )
        # Timer-driven auto-refresh
        timer.tick(
            fn=btn_refresh,
            outputs=[chart_plot, metrics_md]
        )

    demo.launch(
        share=False,
        show_api=False,
        show_error=True,
        inbrowser=True,
        height=900
    )


# ── LAUNCH ───────────────────────────────────────────────────────────────
launch_dashboard()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


---

## Cell 12 — Optional: Save Static Final Plot (no Gradio needed)

If you just want the same PNG as the original notebook without the Gradio UI:

In [ ]:
if EVAL_RESULTS is not None:
    fig_final = build_plot(EVAL_RESULTS, step=None)   # step=None → all data
    out_path  = "AlphaHVAC_Simulation_Final.png"
    fig_final.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig_final)
    print(f"✅ Final static plot saved: {out_path}")
    print(f"   Energy saving  : {EVAL_RESULTS['energy_saving']:.1f}%")
    print(f"   Comfort change : {EVAL_RESULTS['temp_change']:+.1f}%")
    print(f"   Damper mean    : {EVAL_RESULTS['mean_damper']:.3f}")
else:
    print("⚠️  No results to plot.")

✅ Final static plot saved: AlphaHVAC_Simulation_Final.png
   Energy saving  : 62.2%
   Comfort change : +0.0%
   Damper mean    : 0.793


/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/ipykernel_45137/958454828.py:39: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(4, 1, figsize=(14, 16))


: 